### Training a Random Forest Regressor model
Training a random forest regressor that predicts FPL points for the upcoming GW for players. This is a general model, and uses position as one of the predictors. A future step might be to produce a separate model for each position so that position-specific features can be better considered.

Rolling game statistics are key to the model - they will be computed on the previous three games for each player, and used as predictor features.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from model import AdvancedLSTM
import pickle
from eval import season_performance_with_unlimited_transfers

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/backend/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/backend/predictor/processed_data'

In [4]:
X_train = torch.load(data_path + '/X_train.pt', weights_only=True)
y_train = torch.load(data_path + '/y_train.pt', weights_only=True)
train_mapping = pd.read_csv(data_path + '/train_mapping.csv')

X_val = torch.load(data_path + '/X_val.pt', weights_only=True)
y_val = torch.load(data_path + '/y_val.pt', weights_only=True)

X_test = torch.load(data_path + '/X_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')

In [5]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: torch.Size([535271, 5, 49])
y_train shape: torch.Size([535271, 1])
X_val shape: torch.Size([28860, 5, 49])
y_val shape: torch.Size([28860, 1])


In [6]:
def analyze_padding_usage(mapping_df, detailed=True):
    """
    Analyze padding usage in sequence creation based on mapping data.
    
    Parameters:
    -----------
    mapping_df : pandas.DataFrame
        The mapping dataframe returned from create_sequences functions
    detailed : bool, default=True
        Whether to show detailed breakdown by gameweek
    
    Returns:
    --------
    dict : Analysis results including padding statistics
    """
    
    # Basic padding statistics
    padding_stats = {
        'total_sequences': len(mapping_df),
        'sequences_with_padding': (mapping_df['padding_used'] > 0).sum(),
        'sequences_no_padding': (mapping_df['padding_used'] == 0).sum(),
        'max_padding': mapping_df['padding_used'].max(),
        'min_padding': mapping_df['padding_used'].min(),
        'avg_padding': mapping_df['padding_used'].mean(),
    }
    
    # Padding by gameweek
    gw_padding = mapping_df.groupby('prediction_gw')['padding_used'].agg([
        'count', 'mean', 'max', 'min', 
        lambda x: (x > 0).sum(),  # sequences with padding
        lambda x: (x == 0).sum()   # sequences without padding
    ]).round(2)
    
    gw_padding.columns = ['total_sequences', 'avg_padding', 'max_padding', 'min_padding', 
                         'with_padding', 'no_padding']
    
    # Padding distribution
    padding_dist = mapping_df['padding_used'].value_counts().sort_index()
    
    # Players with most padding usage
    player_padding = mapping_df.groupby('name')['padding_used'].agg([
        'count', 'mean', 'max', 'sum'
    ]).round(2)
    player_padding.columns = ['total_predictions', 'avg_padding', 'max_padding', 'total_padding']
    top_padded_players = player_padding.sort_values('avg_padding', ascending=False).head(10)
    
    print("=== PADDING USAGE ANALYSIS ===\n")
    
    print("Overall Statistics:")
    print(f"  Total sequences: {padding_stats['total_sequences']:,}")
    print(f"  Sequences with padding: {padding_stats['sequences_with_padding']:,} ({padding_stats['sequences_with_padding']/padding_stats['total_sequences']*100:.1f}%)")
    print(f"  Sequences without padding: {padding_stats['sequences_no_padding']:,} ({padding_stats['sequences_no_padding']/padding_stats['total_sequences']*100:.1f}%)")
    print(f"  Average padding per sequence: {padding_stats['avg_padding']:.2f}")
    print(f"  Max padding used: {padding_stats['max_padding']}")
    print(f"  Min padding used: {padding_stats['min_padding']}")
    
    print(f"\nPadding Distribution:")
    for padding_amount, count in padding_dist.items():
        percentage = count / len(mapping_df) * 100
        print(f"  {padding_amount} padding steps: {count:,} sequences ({percentage:.1f}%)")
    
    if detailed:
        print(f"\nPadding by Gameweek:")
        print("GW | Total | Avg Pad | Max Pad | With Pad | No Pad | % With Pad")
        print("-" * 65)
        for gw, row in gw_padding.iterrows():
            pct_with_padding = row['with_padding'] / row['total_sequences'] * 100
            print(f"{gw} | {row['total_sequences']:5.0f} | {row['avg_padding']:7.2f} | {row['max_padding']:7.0f} | {row['with_padding']:8.0f} | {row['no_padding']:6.0f} | {pct_with_padding:8.1f}%")
        
        print(f"\nTop 10 Players with Most Padding (by average):")
        print("Player | Total Pred | Avg Pad | Max Pad | Total Pad")
        print("-" * 50)
        for name, row in top_padded_players.iterrows():
            print(f"{name[:15]:15s} | {row['total_predictions']:10.0f} | {row['avg_padding']:7.2f} | {row['max_padding']:7.0f} | {row['total_padding']:9.0f}")
    
    # Analysis insights
    print(f"\n=== KEY INSIGHTS ===")
    
    # Check early gameweek padding
    early_gws = gw_padding.loc[gw_padding.index <= 5]
    late_gws = gw_padding.loc[gw_padding.index > 5]
    
    if len(early_gws) > 0:
        avg_early_padding = early_gws['avg_padding'].mean()
        print(f"Average padding for GW 1-5: {avg_early_padding}")
    
    if len(late_gws) > 0:
        avg_late_padding = late_gws['avg_padding'].mean()
        print(f"Average padding for GW 6+: {avg_late_padding}")

    # Check if padding decreases as expected
    gw_trend = gw_padding['avg_padding'].head(10)  # First 10 gameweeks
    if len(gw_trend) > 1:
        if gw_trend.is_monotonic_decreasing:
            print("✓ Padding correctly decreases as gameweek increases (early GWs)")
        else:
            print("⚠ Padding does not consistently decrease in early gameweeks")
    
    # Check for unexpected late padding
    if len(late_gws) > 0 and late_gws['avg_padding'].max() > 0:
        problematic_gws = late_gws[late_gws['avg_padding'] > 0]
        if len(problematic_gws) > 0:
            print(f"⚠ Unexpected padding found in late gameweeks: {list(problematic_gws.index)}")
    
    return {
        'overall_stats': padding_stats,
        'gameweek_breakdown': gw_padding,
        'padding_distribution': padding_dist,
        'top_padded_players': top_padded_players,
        'mapping_df': mapping_df
    }

In [7]:
# After creating sequences, analyze padding usage

# Analyze overall padding usage for test data
test_padding_analysis = analyze_padding_usage(test_mapping, detailed=True)

=== PADDING USAGE ANALYSIS ===

Overall Statistics:
  Total sequences: 26,499
  Sequences with padding: 3,123 (11.8%)
  Sequences without padding: 23,376 (88.2%)
  Average padding per sequence: 0.29
  Max padding used: 4
  Min padding used: 0

Padding Distribution:
  0 padding steps: 23,376 sequences (88.2%)
  1 padding steps: 780 sequences (2.9%)
  2 padding steps: 781 sequences (2.9%)
  3 padding steps: 781 sequences (2.9%)
  4 padding steps: 781 sequences (2.9%)

Padding by Gameweek:
GW | Total | Avg Pad | Max Pad | With Pad | No Pad | % With Pad
-----------------------------------------------------------------
2.0 |   668 |    4.00 |       4 |      668 |      0 |    100.0%
3.0 |   675 |    3.01 |       4 |      675 |      0 |    100.0%
4.0 |   686 |    2.04 |       4 |      686 |      0 |    100.0%
5.0 |   692 |    1.07 |       4 |      692 |      0 |    100.0%
6.0 |   693 |    0.07 |       4 |       25 |    668 |      3.6%
7.0 |   695 |    0.05 |       4 |       20 |    675 |     

In [8]:
train_padding_analysis = analyze_padding_usage(train_mapping, detailed=True)

=== PADDING USAGE ANALYSIS ===

Overall Statistics:
  Total sequences: 535,271
  Sequences with padding: 347,745 (65.0%)
  Sequences without padding: 187,526 (35.0%)
  Average padding per sequence: 1.46
  Max padding used: 4
  Min padding used: 0

Padding Distribution:
  0 padding steps: 187,526 sequences (35.0%)
  1 padding steps: 98,054 sequences (18.3%)
  2 padding steps: 98,068 sequences (18.3%)
  3 padding steps: 121,645 sequences (22.7%)
  4 padding steps: 29,978 sequences (5.6%)

Padding by Gameweek:
GW | Total | Avg Pad | Max Pad | With Pad | No Pad | % With Pad
-----------------------------------------------------------------
2.0 |  6003 |    4.00 |       4 |     6003 |      0 |    100.0%
3.0 |  6025 |    3.00 |       4 |     6025 |      0 |    100.0%
4.0 |  6037 |    2.01 |       4 |     6037 |      0 |    100.0%
5.0 |  6061 |    1.03 |       4 |     6061 |      0 |    100.0%
6.0 |  6065 |    0.04 |       4 |       93 |   5972 |      1.5%
7.0 | 23947 |    1.50 |       4 |    

In [9]:
best_model_data = torch.load("best_model.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_21161/2890380057.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model.pth"

In [10]:
model = AdvancedLSTM(
    hidden_dim=best_model_data['hidden_dim'],
    num_layers=best_model_data['num_layers'],
    input_dim= 49,
    output_dim=1,
    num_fc_layers=best_model_data['num_fc_layers'],
) 
model.load_state_dict(best_model_data['model_state_dict'])

/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


<All keys matched successfully>

In [11]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

torch.Size([535271, 5, 49])
torch.Size([28860, 5, 49])
torch.Size([26499, 5, 49])


In [12]:
predictions = model(X_test).detach().numpy()
print(predictions.shape)
print(y_test.shape)

(26499, 1)
torch.Size([26499, 1])


In [13]:
test = pd.read_csv(data_path + '/test_data.csv')
test.columns

Index(['GW', 'last_1_assists', 'last_3_assists', 'last_5_assists',
       'last_all_assists', 'last_1_bonus', 'last_3_bonus', 'last_5_bonus',
       'last_all_bonus', 'last_1_bps', 'last_3_bps', 'last_5_bps',
       'last_all_bps', 'last_1_creativity', 'last_3_creativity',
       'last_5_creativity', 'last_all_creativity', 'last_1_clean_sheets',
       'last_3_clean_sheets', 'last_5_clean_sheets', 'last_all_clean_sheets',
       'last_1_goals_conceded', 'last_3_goals_conceded',
       'last_5_goals_conceded', 'last_all_goals_conceded',
       'last_1_goals_scored', 'last_3_goals_scored', 'last_5_goals_scored',
       'last_all_goals_scored', 'last_1_ict_index', 'last_3_ict_index',
       'last_5_ict_index', 'last_all_ict_index', 'last_1_influence',
       'last_3_influence', 'last_5_influence', 'last_all_influence',
       'last_1_minutes', 'last_3_minutes', 'last_5_minutes',
       'last_all_minutes', 'last_1_threat', 'last_3_threat', 'last_5_threat',
       'last_all_threat', 'last_1

In [14]:
remaining_lagged_features = pickle.load(open(data_path + '/remaining_lagged_features.pkl', 'rb'))

In [15]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=remaining_lagged_features
)

/Users/bragehs/Documents/FPL_forecast/backend/predictor/eval.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions_df_complete = pd.concat([gameweek_1, predictions_df], ignore_index=True)


Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
0.0 :  lukasz_fabianski
1.0 :  sepp_van_den_berg
2.0 :  tyler_dibling
3.0 :  daniel_jebbison
Bench players: ['lukasz_fabianski', 'sepp_van_den_berg', 'tyler_dibling', 'daniel_jebbison']
Bench cost: 170
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 830

--- Gameweek 1.0 ---
Players available for GW 1.0: 0
Available cash: 830
Decision variables: [aaron_anselmino, aaron_cresswell, aaron_hickey, aaron_ramsdale, aaron_wan_bissaka, abdoulaye_doucoure, abdukodir_khusanov, abdul_fatawu, adam_armstrong, adam_lallana, adam_smith, adam_webster, adam_wharton, adama_traore, adrian_mazilu, albert_gronbaek, alejandro_garnacho, alejo_veliz, alex_iwobi, alex_mccarthy, alex_mighten, alex_moreno_lopera, alex_murphy, alex_palmer, alex_paulsen, alex_scott, alexander_isak, alexis_mac_allister, alfie_devine, alfie_dorrington, alfie_gilchrist, alfie_pond, alfie_whiteman, ali_al_hamadi, 

In [16]:
total_score.item()

1849.0

In [17]:
scores

,team,gw_score
0,"[anthony_gordon, benjamin_white, cole_palmer, ...",48.0
1,"[alexander_isak, alisson_ramses_becker, andrew...",66.0
2,"[alexander_isak, benjamin_white, bruno_borges_...",74.0
3,"[andreas_hoelgebaum_pereira, andrew_robertson,...",66.0
4,"[alisson_ramses_becker, bukayo_saka, dominik_s...",62.0
5,"[antonee_robinson, bukayo_saka, chris_wood, da...",47.0
6,"[amad_diallo, dan_burn, erling_haaland, levi_c...",37.0
7,"[andreas_hoelgebaum_pereira, antonee_robinson,...",42.0
8,"[adama_traore, ashley_young, declan_rice, erli...",50.0
9,"[ashley_young, bernardo_veiga_de_carvalho_e_si...",30.0
